<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">

# Part 2 — World Reasoning with Cosmos 3

**Cosmos 3** is an open *omni-model* for Physical AI — a single model that can reason about, generate, and act in the physical world. This notebook uses its **reasoning** capability: Cosmos 3 as a vision-language model (VLM) that brings physical common sense, spatial and temporal understanding, and long chain-of-thought reasoning to images and video.

For robotics and embodied AI, a capable reasoner is the "perception and planning" brain: before a robot can act, it has to *understand* what it is looking at, reason about what is physically possible, decide what to do next, and localize the objects it will interact with. In this notebook we explore this reasoning capability on robot and physical-interaction footage.

The reasoner is served here as an **NVIDIA NIM** microservice with an OpenAI-compatible API, so we talk to it with the standard `openai` Python client. It ships in two sizes — **Nano (8B)** for single-GPU deployment and **Super (32B)** for higher accuracy; this lab uses Nano.

**Capabilities we will try (all on robotics / physical-interaction footage):**

| Capability | What it answers | Why it matters for Physical AI |
| --- | --- | --- |
| **Video captioning** | *What is happening in this clip?* | Turns raw video into structured scene descriptions |
| **Embodied reasoning** | *What should the agent do next? What is the plan?* | Bridges perception to action |
| **Common-sense reasoning** | *Is this physically sensible?* | Grounds decisions in real-world physics |
| **2D grounding** | *Where exactly is object X?* | Localizes objects to interact with |
| **Describe anything** | *Describe each marked region.* | Fine-grained, region-level understanding |
| **Action chain-of-thought** | *What path should the gripper follow?* | Produces spatial plans for manipulation |

## How this notebook fits the pipeline

Cosmos 3 powers an end-to-end **perceive → reason → act** loop:

1. **Reason (this notebook):** understand robot scenes — caption them, localize objects, judge physical plausibility, and plan the next action.
2. **Generate (Part 3):** turn scene descriptions into new, photorealistic training video.
3. **Act (Part 4):** predict the actions and trajectories that carry out a task.

The captions, plans, groundings, and trajectories produced here are signals that the generation and action stages can consume — so strengthening the reasoner strengthens the whole loop.

## Setup

In [ ]:
import os
import sys
import base64
import mimetypes
import subprocess
from pathlib import Path
from IPython.display import Image, Video, display

# The Cosmos 3 cookbook ships the robotics sample
# media used here. These files are already H.264 / PNG, so they render inline.
ASSETS_DIR = Path("/opt/cosmos/cookbooks/cosmos3/reasoner/assets")
ASSETS_DIR_2 = Path("/dli/task/assets")

ASSETS = {
    "video_caption":        ASSETS_DIR / "video_caption.mp4",
    "robotics_next_action": ASSETS_DIR / "robotics_next_action.mp4",
    "robot_planning":       ASSETS_DIR / "robot_planning.png",
    "assisted_task":        ASSETS_DIR_2 / "toaster_6sec.mp4",
    "common_sense":         ASSETS_DIR / "common_sense_reasoning.mp4",
    "grounding_2d":         ASSETS_DIR / "grounding_2d.png",
    "robot_workspace":      ASSETS_DIR / "robot_153.jpg",
    "action_cot":           ASSETS_DIR / "action_cot_trajectory.png",
    "situation":            ASSETS_DIR / "situation_understanding.mp4",
}
for name, path in ASSETS.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing asset '{name}': {path}")

# The OpenAI Python client is used to talk to the NIM's OpenAI-compatible API.
try:
    import openai  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai"], check=True)
    import openai  # noqa: F401

# Everything this notebook produces (transcodes, marked images, overlays, answers)
# goes under <repo>/outputs/notebook2.
OUTPUT_ROOT = Path(os.environ.get("DLI_OUTPUTS", Path("/dli/task").resolve().parent / "outputs")) / "notebook2"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Environment ready.")
print(f"  Assets: {ASSETS_DIR}")
print(f"  Outputs: {OUTPUT_ROOT}")
print(f"  openai client: {openai.__version__}")

## Connecting to the Cosmos 3 Reasoner NIM

The reasoner runs as an **NVIDIA NIM** service launched alongside this lab by Docker Compose (the `cosmos3-reasoner` service). It exposes an OpenAI-compatible API and is reachable over the Compose network at `http://cosmos3-reasoner:8000` — so you do not have to run a separate `docker run` step.

Media (videos and images) is sent to the model **inline as base64** in each request, so the NIM does not need the assets mounted to see them.

In [ ]:
NIM_BASE_URL = os.environ.get("NIM_BASE_URL", "http://cosmos3-reasoner:8000")
print(f"Cosmos 3 Reasoner NIM: {NIM_BASE_URL}")

In [ ]:
# Wait for the NIM to become ready (first start downloads weights).
import requests
import time

ready_url = f"{NIM_BASE_URL}/v1/health/ready"
models_url = f"{NIM_BASE_URL}/v1/models"
max_wait = 1200

print(f"Waiting for the NIM at {NIM_BASE_URL} ...")
start = time.time()
while time.time() - start < max_wait:
    try:
        if requests.get(ready_url, timeout=2).status_code == 200:
            print(f"\nNIM ready after {int(time.time() - start)}s.")
            data = requests.get(models_url, timeout=5).json().get("data", [])
            if data:
                print(f"  Serving model: {data[0]['id']}")
            break
    except requests.exceptions.RequestException:
        pass
    print(".", end="", flush=True)
    time.sleep(5)
else:
    raise TimeoutError("NIM did not become ready. Check it from a host terminal: "
                       "docker compose -f task1/docker-compose.yml logs cosmos3-reasoner")

### The inference helper

The `run_inference` helper below wraps the OpenAI client so each example stays short. Multimodal inputs are passed as message content parts (`image_url`, `video_url`, `text`); because the NIM does not read local file paths, the helper inlines local media as base64 `data:` URLs. It prints and returns the model's text. To preview the input clip/image, call the separate `show_media` helper (which accepts local paths or URLs for both images and videos).

Useful knobs:
- `fps` — how densely attached videos are sampled into frames (sent via `media_io_kwargs`).
- `max_tokens` — response length cap (we use 4096 to leave room for reasoning traces).
- `temperature` / `top_p` / `top_k` / `repetition_penalty` / `seed` — sampling controls (omitted → server defaults; `seed` makes a run reproducible).

Several prompts ask the model to "think" using a `<think> … </think>` block followed by a final answer — this exposes the model's **chain of thought** so you can inspect *why* it reached a conclusion, which is essential when the output will drive a robot.

> **Bring your own video:** the NIM decodes video with OpenCV, which cannot read some codecs (e.g. AV1). `run_inference` automatically transcodes any non-H.264 clip to H.264 (cached) before sending it, so you can point the examples at your own footage.

In [ ]:
from openai import OpenAI
import json
import subprocess

# The OpenAI-compatible API ignores the key, but the client requires a value.
client = OpenAI(base_url=f"{NIM_BASE_URL}/v1", api_key="not-used")

# Resolve the served model name dynamically (nano or super), with a fallback.
try:
    MODEL_ID = client.models.list().data[0].id
except Exception:
    MODEL_ID = "nvidia/cosmos3-nano-reasoner"
print(f"Using model: {MODEL_ID}")

# The NIM decodes video with OpenCV, which cannot read some codecs (notably AV1
# and some HEVC). Such clips return metadata but 0 decoded frames, which the
# server rejects with a 422 error. So the helper below transcodes any non-H.264
# video to H.264/yuv420p once (cached) before sending it, letting you drop in
# arbitrary clips without worrying about their codec.
_H264_CACHE = OUTPUT_ROOT / "h264_cache"


def _video_codec(path):
    """Return the video stream's codec name via ffprobe, or None if unavailable."""
    try:
        return subprocess.run(
            ["ffprobe", "-v", "error", "-select_streams", "v:0",
             "-show_entries", "stream=codec_name", "-of", "default=nk=1:nw=1", str(path)],
            capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None


def ensure_h264(video):
    """Return a path the NIM can decode. H.264 (and remote/inline) inputs pass
    through unchanged; other codecs are transcoded to H.264/yuv420p once and cached."""
    s = str(video)
    if s.startswith(("http://", "https://", "data:")):
        return video
    codec = _video_codec(s)
    if codec in (None, "h264"):
        return video  # already H.264, or ffprobe unavailable -> try as-is
    _H264_CACHE.mkdir(parents=True, exist_ok=True)
    dst = _H264_CACHE / f"{Path(s).stem}_h264.mp4"
    if not dst.exists() or dst.stat().st_mtime < Path(s).stat().st_mtime:
        print(f"Transcoding {Path(s).name} ({codec} -> h264) so the NIM can decode it ...")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", s,
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", str(dst)], check=True)
    return dst


def _to_url(path_or_url):
    """Return a URL the NIM can consume, inlining local files as base64 data URLs."""
    s = str(path_or_url)
    if s.startswith(("http://", "https://", "data:")):
        return s
    mime = mimetypes.guess_type(s)[0] or "application/octet-stream"
    data = base64.b64encode(Path(s).read_bytes()).decode("ascii")
    return f"data:{mime};base64,{data}"


def show_media(images=None, videos=None, media_width=512):
    """Display images and videos inline. Each item may be a local file path or
    an http(s) URL (or base64 data URL); both are handled transparently."""
    for p in (images or []):
        s = str(p)
        if s.startswith(("http://", "https://", "data:")):
            display(Image(url=s, width=media_width))
        else:
            display(Image(filename=s, width=media_width))
    for p in (videos or []):
        s = str(p)
        if s.startswith(("http://", "https://", "data:")):
            display(Video(url=s, width=media_width))
        else:
            display(Video(s, embed=True, width=media_width))


def run_inference(prompt, *, images=None, videos=None, fps=4.0, max_tokens=4096,
                  temperature=None, top_p=None, repetition_penalty=None,
                  top_k=None, presence_penalty=None, seed=None):
    images = images or []
    # Transcode non-H.264 clips (e.g. AV1) so the NIM's decoder can read them.
    videos = [ensure_h264(v) for v in (videos or [])]

    content = []
    for p in images:
        content.append({"type": "image_url", "image_url": {"url": _to_url(p)}})
    for p in videos:
        content.append({"type": "video_url", "video_url": {"url": _to_url(p)}})
    content.append({"type": "text", "text": prompt})

    extra_body = {}
    if videos:
        extra_body["media_io_kwargs"] = {"video": {"fps": float(fps)}}
    if repetition_penalty is not None:
        extra_body["repetition_penalty"] = repetition_penalty
    if top_k is not None:
        extra_body["top_k"] = top_k

    kwargs = dict(model=MODEL_ID, messages=[{"role": "user", "content": content}], max_tokens=max_tokens)
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p
    if presence_penalty is not None:
        kwargs["presence_penalty"] = presence_penalty
    if seed is not None:
        kwargs["seed"] = seed
    if extra_body:
        kwargs["extra_body"] = extra_body

    answer = client.chat.completions.create(**kwargs).choices[0].message.content
    print(answer)
    # Keep a log of every prompt/answer pair alongside the other outputs.
    with open(OUTPUT_ROOT / "responses.jsonl", "a") as f:
        f.write(json.dumps({"prompt": prompt, "images": [str(p) for p in images],
                            "videos": [str(v) for v in videos], "answer": answer}) + "\n")
    return answer

def display_video(path, width=640):
    data = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    display(HTML(f'<video controls playsinline width="{width}" style="background:#000">'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
                 f'<div style="font-family:monospace;font-size:12px">{html.escape(str(path))}</div>'))

### Overlay helpers

Two capabilities return *spatial* answers — bounding boxes and gripper trajectories — as JSON with coordinates normalized to a 0–1000 range. The helpers below parse that JSON and draw it back onto the image so we can see where the model is pointing.

In [ ]:
import json
import re
from PIL import Image as PILImage, ImageDraw


def _parse_json(text, after_think=False):
    """Extract the first JSON array/object from model text (handles ``` fences)."""
    if after_think and "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip().strip("`").strip()
    match = re.search(r"\[.*\]|\{.*\}", text, re.DOTALL)
    data = json.loads(match.group(0) if match else text)
    return data if isinstance(data, list) else [data]


def draw_boxes(image_path, text, width=768):
    """Draw predicted bounding boxes (coords normalized to 0-1000) on the image."""
    try:
        objs = _parse_json(text)
    except Exception as exc:
        print("Could not parse boxes from the response:", exc)
        return
    img = PILImage.open(image_path).convert("RGB")
    W, H = img.size
    draw = ImageDraw.Draw(img)
    for obj in objs:
        box = obj.get("bbox_2d") or obj.get("bbox") or obj.get("box")
        if not box:
            continue
        x1, y1, x2, y2 = box
        x1, x2 = x1 / 1000 * W, x2 / 1000 * W
        y1, y2 = y1 / 1000 * H, y2 / 1000 * H
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        label = obj.get("label") or obj.get("name")
        if label:
            draw.text((x1, max(0, y1 - 12)), str(label), fill="red")
    out = OUTPUT_ROOT / f"{Path(image_path).stem}_boxes.png"
    img.save(out)
    print("saved:", out)
    preview = img.copy()
    preview.thumbnail((width, width))
    display(preview)


def draw_trajectory(image_path, text, width=900):
    """Draw a predicted 2D gripper trajectory (coords normalized to 0-1000)."""
    try:
        objs = _parse_json(text, after_think=True)
    except Exception as exc:
        print("Could not parse trajectory from the response:", exc)
        return
    img = PILImage.open(image_path).convert("RGB")
    W, H = img.size
    draw = ImageDraw.Draw(img)
    pts = [(o["point_2d"][0] / 1000 * W, o["point_2d"][1] / 1000 * H)
           for o in objs if isinstance(o, dict) and "point_2d" in o]
    if len(pts) > 1:
        draw.line(pts, fill="lime", width=5)
    for i, (x, y) in enumerate(pts):
        r = 12
        draw.ellipse([x - r, y - r, x + r, y + r], fill="red", outline="white", width=3)
        draw.text((x + 14, y - 14), str(i), fill="yellow")
    out = OUTPUT_ROOT / f"{Path(image_path).stem}_trajectory.png"
    img.save(out)
    print("saved:", out)
    preview = img.copy()
    preview.thumbnail((width, width))
    display(preview)

print("overlay helpers ready")

# 1. Video captioning

Captioning is the foundation of scene understanding: the model samples the video into frames (here at 4 FPS), and produces a detailed description of the setting, the objects, and the sequence of actions. For a data pipeline, these captions become the structured record of "what happened" — and, lightly edited, the prompts that drive video generation.

In [ ]:
input_media = ASSETS["video_caption"]
# input_media = "https://nvidia-cosmos.github.io/cosmos-cookbook/recipes/post_training/reason2/video_caption_vqa/assets/1a2a96fe-1896-4b2c-9f87-3bbdadd27920.camera_front_wide_120fov.mp4"
# input_media = "https://nvidia-cosmos.github.io/cosmos-cookbook/recipes/end2end/gr00t-dreams/assets/1.mp4"

show_media(videos=[input_media])

In [ ]:
_ = run_inference("Describe the video in detail.",
                  videos=[input_media], fps=4, max_tokens=4096)

# 2. Embodied reasoning

Embodied reasoning is where perception meets action: given what it sees, the model reasons about **what the agent should do next** and **how to break a goal into steps**. We ask the model to expose its chain of thought in a `<think> … </think>` block so its decision is auditable before anything drives a robot.

## 2a. Robotics next action

Given a clip of a robot mid-task, predict the next immediate action. This is the core query of a reactive policy: *given the current observation, what do I do now?*

In [ ]:
prompt = (
    "What can be the next immediate action? Answer the question using the following "
    "format: <think> Your reasoning. </think> Write your final answer immediately "
    "after the </think> tag."
)

input_media = ASSETS["robotics_next_action"]
show_media(videos=[input_media])
_ = run_inference(prompt, videos=[input_media], fps=4, max_tokens=4096)

## 2b. Robot planning

Higher-level than a single action: decompose a goal into an ordered list of subtasks. Task planning like this is what lets a robot carry out a *described* objective rather than a fixed, pre-scripted motion.

In [ ]:
prompt = ("The task is to put flower into the red bottle. Generate a plan consisting "
          "of subtasks for accomplish the task.")

input_media = ASSETS["robot_planning"]
show_media(images=[input_media])
_ = run_inference(prompt, images=[input_media], max_tokens=4096, seed=0)

## 2c. Assisted-task next action

The model is given an overall goal *and* the current step, and must decide the next action — the pattern behind an AI assistant that guides a person (or robot) through a multi-step procedure.

In [ ]:
prompt = """This is the overall task that the agent is trying to complete: "The robot needs to toast the bread"
In the video, the agent is trying to follow the instruction (a single step out of many to complete the overall task): "toast the bread_slice."
What should be the next action of the agent?
Answer the question using the following format:
<think>
Your reasoning.
</think>
Write your final answer immediately after the </think> tag."""

input_media = ASSETS["assisted_task"]
show_media(videos=[input_media])
_ = run_inference(prompt, videos=[input_media], fps=4, max_tokens=4096)

# 3. Common-sense reasoning

Physical common sense keeps decisions grounded in reality — an agent must know, for example, whether a surface can bear a load before placing something on it. Here the model reasons about feasibility from the video before answering.

In [ ]:
prompt = """Can the countertop support the weight of the toaster?
Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag."""

input_media = ASSETS["assisted_task"]
show_media(videos=[input_media])
_ = run_inference(prompt, videos=[input_media], fps=4, max_tokens=4096)

# 4. 2D grounding

To interact with an object, a robot first has to **localize** it. Grounding returns a bounding box for a referenced object; the model outputs JSON with coordinates normalized to 0–1000, which we scale to pixels and draw on the image.

In [ ]:
input_media = ASSETS["robot_planning"]
show_media(images=[input_media])

out = run_inference("Locate the accurate bounding box of the cameras. Return a json.",
                    images=[input_media], max_tokens=4096)
draw_boxes(ASSETS["robot_planning"], out)

# 5. Describe anything

*Describe Anything* is region-level understanding driven by **Set-of-Mark** prompting: rather than captioning the whole scene, we overlay **numbered marks** on the subjects we care about and ask the model to describe **each marked subject**. The `subject_id` in the response refers back to the number drawn on the image, so every caption is anchored to a specific region — exactly the structured, object-level record you would build from a robot's camera feed to reason about what can be grasped or moved.

Below we mark a curated set of salient subjects in a **robot manipulation workspace** — the two robot arms, the person, the camera tripod, and the red ball — then let the model describe each one, keyed by its mark.

> In a full pipeline the region masks come from a segmentation model (e.g. SAM); here we mark the regions directly so the numbering is stable and reproducible.

In [ ]:
# "Describe Anything" is a *Set-of-Mark* task: the model describes the subjects
# that are explicitly MARKED in the image, and the "subject_id" it returns refers
# back to those marks. So we first overlay numbered marks on the salient subjects,
# then ask the model to describe each marked subject.
from PIL import Image as PILImage, ImageDraw, ImageFont

# Salient subjects in robot_153.jpg (1280x720), as pixel bounding boxes.
MARKS = [
    {"id": 1, "bbox": [300, 430, 480, 700]},   # left robotic arm / gripper
    {"id": 2, "bbox": [820, 420, 990, 700]},   # right robotic arm
    {"id": 3, "bbox": [895,  35, 1010, 120]},   # red apple on the desk
    {"id": 4, "bbox": [270,   0, 465, 215]},   # person standing
]

_MARK_COLORS = [(255,59,48),(0,122,255),(52,199,89),(255,149,0),(175,82,222),(255,45,146)]


def mark_subjects(image_path, marks, out_path, badge=36):
    """Overlay numbered Set-of-Mark annotations (a translucent region + an ID
    badge) so the model can describe each subject by its mark."""
    img = PILImage.open(image_path).convert("RGBA")
    overlay = PILImage.new("RGBA", img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", int(badge * 0.6))
    except Exception:
        font = ImageFont.load_default()
    for m in marks:
        color = _MARK_COLORS[(m["id"] - 1) % len(_MARK_COLORS)]
        x1, y1, x2, y2 = m["bbox"]
        draw.rectangle([x1, y1, x2, y2], fill=color + (60,), outline=color + (255,), width=4)
        # numbered badge in the top-left corner of the marked region
        bx, by = x1, y1
        draw.ellipse([bx, by, bx + badge, by + badge], fill=color + (255,), outline=(255, 255, 255, 255), width=3)
        t = str(m["id"])
        tb = draw.textbbox((0, 0), t, font=font)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]
        draw.text((bx + (badge - tw) / 2 - tb[0], by + (badge - th) / 2 - tb[1]),
                  t, fill=(255, 255, 255, 255), font=font)
    out = PILImage.alpha_composite(img, overlay).convert("RGB")
    out.save(out_path)
    return out_path


MARKED_WORKSPACE = str(OUTPUT_ROOT / "robot_workspace_marked.png")
mark_subjects(ASSETS["robot_workspace"], MARKS, MARKED_WORKSPACE)

# Canonical describe-anything prompt: describe each *marked* subject, keyed by id.
prompt = ('Please caption the notable attributes in the provided image. List and '
          'describe all marked subjects in the image with their categories and '
          'detailed captions using a json with keyword "subject_id", "category" '
          'and "caption".')

show_media(images=[MARKED_WORKSPACE])
_ = run_inference(prompt, images=[MARKED_WORKSPACE], max_tokens=4096, seed=0)

# 6. Action chain-of-thought (gripper trajectory)

Beyond *what* to do, the model can reason about *how* to move: given a manipulation task, it plans the 2D path the end effector should follow in pixel space and returns the waypoints as JSON. We overlay that trajectory on the image. This uses slightly higher sampling temperature to encourage a smooth, exploratory path.

In [ ]:
prompt = """You are given the task "Move the pink bowl to the right". Specify the 2D trajectory your end effector should follow in pixel space. Return the trajectory coordinates in JSON format like this: {"point_2d": [x, y], "label": "gripper trajectory"}.
Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag."""

input_media = ASSETS["action_cot"]
show_media(images=[input_media])

out = run_inference(prompt, images=[input_media], max_tokens=4096,
                    temperature=0.6, top_p=0.95, presence_penalty=0.0,
                    top_k=20, repetition_penalty=1.0)
draw_trajectory(input_media, out)

## Wrapping up

You have explored the Cosmos 3 reasoner across the reasoning skills a Physical AI system relies on: **captioning** scenes, **planning** and predicting **next actions**, applying **physical common sense**, **grounding** and **describing** objects, plotting **manipulation trajectories**, and **anticipating** what happens next — all on robotics and physical-interaction footage.

These outputs feed the rest of the loop: the captions and plans can be used as prompts for **world generation**, and the next-action and trajectory reasoning can be used for **action generation**.

## Resources

- [Cosmos 3 Reasoner NIM (NGC)](https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/cosmos3-reasoner)
- [Try the reasoner in your browser (build.nvidia.com)](https://build.nvidia.com/nvidia/cosmos3-nano-reasoner)
- [Cosmos 3 cookbook — Reasoner](https://github.com/NVIDIA/cosmos/tree/main/cookbooks/cosmos3/reasoner)

<br clear="all">
<hr>
<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">